# Knox County City2Graph — Step 1 & Step 2
## Build Base Knox Datasets + Build Morphology and OD Graphs

**Goal:** Load, clean, align, and validate all Knox County data layers, then build two separate graphs:
- (a) OD Graph from OD matrix + zones
- (b) Morphology Graph from road network + buildings (OSM fallback)

**Data Sources:**
- Zones: `KnoxTPO_Network_TAZShapefiles_OD/Knox_TAZ_shapefile.shp`
- Road Links: `KnoxTPO_Network_TAZShapefiles_OD/Links_shapefile.shp`
- OD Matrix: `KnoxTPO_Network_TAZShapefiles_OD/2026_matrix to excel.xlsx`
- Assignment: `Knox_Network_w_Attributes_Assignment/Knox_Network w Attributes.shp`
- Assignment Volumes: `Knox_Network_w_Attributes_Assignment/2026_LinkFlows.xlsx`

---
## Section 1: Import Libraries and Define Paths

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ── Data root paths (local copies inside this project) ───────────────────────
TAZ_DIR    = Path(r"C:\Users\ets\Desktop\GithubProjects\KnoxCity2Graph\data\KnoxTPO_Network_TAZShapefiles_OD")
ASSIGN_DIR = Path(r"C:\Users\ets\Desktop\GithubProjects\KnoxCity2Graph\data\Knox_Network_w_Attributes_Assignment")

# ── Individual file paths ─────────────────────────────────────────────────────
ZONES_SHP   = TAZ_DIR    / "Knox_TAZ_shapefile.shp"
LINKS_SHP   = TAZ_DIR    / "Links_shapefile.shp"
OD_XLSX     = TAZ_DIR    / "2026_matrix to excel.xlsx"
ASSIGN_SHP  = ASSIGN_DIR / "Knox_Network w Attributes.shp"
ASSIGN_XLSX = ASSIGN_DIR / "2026_LinkFlows.xlsx"

# ── Target projected CRS (UTM zone 17N – covers Knox County TN) ──────────────
TARGET_CRS = "EPSG:26917"

# ── Verify all files exist ────────────────────────────────────────────────────
for f in [ZONES_SHP, LINKS_SHP, OD_XLSX, ASSIGN_SHP, ASSIGN_XLSX]:
    status = "✓" if f.exists() else "✗ MISSING"
    print(f"  {status}  {f.name}")

print("\nPaths configured. Ready to load data.")


: 

---
## Section 2: Load All Data Layers and Inspect Columns

In [ ]:
# ── Load shapefiles ───────────────────────────────────────────────────────────
zones_gdf   = gpd.read_file(ZONES_SHP)
links_gdf   = gpd.read_file(LINKS_SHP)
assign_gdf  = gpd.read_file(ASSIGN_SHP)

# ── Load tabular data ─────────────────────────────────────────────────────────
od_raw      = pd.read_excel(OD_XLSX, index_col=0)
assign_vol  = pd.read_excel(ASSIGN_XLSX)

print("=== ZONES columns ===")
print(zones_gdf.columns.tolist())
print(zones_gdf.shape, "| CRS:", zones_gdf.crs)
print(zones_gdf.head(2))

print("\n=== LINKS columns ===")
print(links_gdf.columns.tolist())
print(links_gdf.shape, "| CRS:", links_gdf.crs)
print(links_gdf.head(2))

print("\n=== ASSIGNMENT NETWORK columns ===")
print(assign_gdf.columns.tolist())
print(assign_gdf.shape, "| CRS:", assign_gdf.crs)
print(assign_gdf.head(2))

print("\n=== OD MATRIX shape ===")
print(od_raw.shape)
print(od_raw.iloc[:3, :5])

print("\n=== ASSIGNMENT VOLUMES columns ===")
print(assign_vol.columns.tolist())
print(assign_vol.shape)
print(assign_vol.head(2))


---
## Section 3: Reproject All Layers to Common CRS (EPSG:26917)

In [ ]:
zones_gdf  = zones_gdf.to_crs(TARGET_CRS)
links_gdf  = links_gdf.to_crs(TARGET_CRS)
assign_gdf = assign_gdf.to_crs(TARGET_CRS)

print("CRS after reprojection:")
print("  zones :", zones_gdf.crs)
print("  links :", links_gdf.crs)
print("  assign:", assign_gdf.crs)


---
## Section 4: Assign Unique zone_id and Melt OD Matrix to Long Format

In [ ]:
# ── Inspect zone ID candidates ────────────────────────────────────────────────
print("Zone candidate ID columns:")
print(zones_gdf.dtypes)
print(zones_gdf[zones_gdf.columns[:6]].head(5))


In [ ]:
# ── Assign zone_id ────────────────────────────────────────────────────────────
# Adjust 'TAZ' below if the actual column name is different (run cell above first)
ZONE_ID_COL = "TAZ"   # <-- UPDATE if needed after inspecting column names above

if ZONE_ID_COL not in zones_gdf.columns:
    # Fallback: use first integer column
    int_cols = [c for c in zones_gdf.columns if zones_gdf[c].dtype in [int, "int64", "int32"]]
    ZONE_ID_COL = int_cols[0]
    print(f"ZONE_ID_COL not found — using fallback: {ZONE_ID_COL}")

zones_gdf = zones_gdf.rename(columns={ZONE_ID_COL: "zone_id"})
zones_gdf["zone_id"] = zones_gdf["zone_id"].astype(int)

print(f"zone_id dtype: {zones_gdf['zone_id'].dtype}")
print(f"Unique zones: {zones_gdf['zone_id'].nunique()}")
print(f"Any duplicates: {zones_gdf['zone_id'].duplicated().any()}")
print(zones_gdf[["zone_id", "geometry"]].head(3))


In [ ]:
# ── Melt OD matrix to long format ────────────────────────────────────────────
od_raw.index = od_raw.index.astype(int)
od_raw.columns = od_raw.columns.astype(int)

od_long = (
    od_raw
    .stack()
    .reset_index()
    .rename(columns={"level_0": "origin", "level_1": "destination", 0: "flow"})
)
od_long = od_long[od_long["flow"] > 0].reset_index(drop=True)

print(f"Total OD pairs (flow > 0): {len(od_long):,}")
print(f"Unique origins:            {od_long['origin'].nunique()}")
print(f"Unique destinations:       {od_long['destination'].nunique()}")
print(f"Total flow:                {od_long['flow'].sum():,.0f}")
print(od_long.head(5))


---
## Section 5: Join Assignment Volumes to Road Network

In [ ]:
# Inspect assignment volume file columns
print("Assignment volume columns:")
print(assign_vol.columns.tolist())
print(assign_vol.head(3))

print("\nAssignment network columns:")
print(assign_gdf.columns.tolist())
print(assign_gdf.head(3))


In [ ]:
# ── Join assignment volumes to network shapefile ──────────────────────────────
# Adjust LINK_KEY to the shared join column (e.g., 'IID', 'LINKID', 'ID', 'AB_GV_VOL')
# Run the inspect cell above first to confirm the exact column name.

ASSIGN_KEY = "IID"   # <-- UPDATE after inspecting columns above

if ASSIGN_KEY in assign_gdf.columns and ASSIGN_KEY in assign_vol.columns:
    assign_merged = assign_gdf.merge(assign_vol, on=ASSIGN_KEY, how="left")
    n_joined = assign_merged["AB_GV_VOL"].notna().sum() if "AB_GV_VOL" in assign_merged.columns else assign_merged.iloc[:, -1].notna().sum()
    print(f"Assignment links total:          {len(assign_gdf):,}")
    print(f"Links with assignment volumes:   {n_joined:,}")
    print(f"Links missing volumes:           {len(assign_gdf) - n_joined:,}")
    print(assign_merged.head(3))
else:
    print(f"Join key '{ASSIGN_KEY}' not found in one or both files.")
    print("assign_gdf keys:", assign_gdf.columns.tolist())
    print("assign_vol keys:", assign_vol.columns.tolist())
    assign_merged = assign_gdf.copy()


---
## Section 6: Step 1 Validation Table

In [ ]:
# ── Validate OD zone IDs match zone_id ───────────────────────────────────────
valid_zone_ids = set(zones_gdf["zone_id"].unique())

od_origins_valid      = od_long["origin"].isin(valid_zone_ids)
od_dest_valid         = od_long["destination"].isin(valid_zone_ids)
od_both_valid         = od_origins_valid & od_dest_valid

pct_od_valid = od_both_valid.sum() / len(od_long) * 100

# ── Summarise links joined to volumes ────────────────────────────────────────
try:
    vol_col = [c for c in assign_merged.columns if "VOL" in c.upper() or "VOLUME" in c.upper()]
    n_links_with_vol = assign_merged[vol_col[0]].notna().sum() if vol_col else 0
except:
    n_links_with_vol = 0

# ── Validation table ──────────────────────────────────────────────────────────
val_df = pd.DataFrame({
    "Metric": [
        "Number of zones",
        "Number of OD pairs (flow > 0)",
        "Number of road links (TAZ folder)",
        "Number of assignment links",
        "Assignment links with volumes",
        "OD rows with valid zone IDs (%)"
    ],
    "Value": [
        zones_gdf["zone_id"].nunique(),
        len(od_long),
        len(links_gdf),
        len(assign_gdf),
        n_links_with_vol,
        f"{pct_od_valid:.1f}%"
    ]
})

print("=" * 50)
print("  STEP 1 VALIDATION TABLE")
print("=" * 50)
print(val_df.to_string(index=False))
print("=" * 50)

# ── Flag mismatched OD IDs ────────────────────────────────────────────────────
bad_origins = od_long[~od_origins_valid]["origin"].unique()
bad_dests   = od_long[~od_dest_valid]["destination"].unique()
if len(bad_origins) > 0:
    print(f"\n⚠ OD origins not matching any zone_id: {bad_origins[:10]}")
if len(bad_dests) > 0:
    print(f"⚠ OD destinations not matching any zone_id: {bad_dests[:10]}")
else:
    print("\n✓ All OD origins and destinations match zone IDs.")


---
## Section 7: Step 2a — Build the OD Graph

Compute zone-level productions, attractions, and weighted degree from the OD table.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# ── Zone-level OD summaries ───────────────────────────────────────────────────
productions  = od_long.groupby("origin")["flow"].sum().rename("production")
attractions  = od_long.groupby("destination")["flow"].sum().rename("attraction")
out_degree   = od_long.groupby("origin")["destination"].count().rename("out_degree")
in_degree    = od_long.groupby("destination")["origin"].count().rename("in_degree")

od_zone_stats = (
    zones_gdf[["zone_id"]]
    .set_index("zone_id")
    .join(productions, how="left")
    .join(attractions, how="left")
    .join(out_degree, how="left")
    .join(in_degree, how="left")
    .fillna(0)
)
od_zone_stats["weighted_degree"] = od_zone_stats["out_degree"] + od_zone_stats["in_degree"]

print("OD Zone Stats (top 10 by production):")
print(od_zone_stats.sort_values("production", ascending=False).head(10))

# ── Build NetworkX OD graph ───────────────────────────────────────────────────
G_od = nx.DiGraph()
for _, row in od_long.iterrows():
    G_od.add_edge(int(row["origin"]), int(row["destination"]), weight=row["flow"])

print(f"\nOD Graph — Nodes: {G_od.number_of_nodes():,}  |  Edges: {G_od.number_of_edges():,}")

# ── Top 20 strongest OD pairs ─────────────────────────────────────────────────
top_od = od_long.sort_values("flow", ascending=False).head(20)
print("\nTop 20 OD pairs by flow:")
print(top_od.to_string(index=False))


---
## Section 8: Step 2b — Build Morphology Graph (Road Network + OSM Buildings)

Compute zone-level morphology features: street density, segment length, building density.

In [ ]:
import osmnx as ox

# ── Fetch Knox County road network from OSM (for morphology) ─────────────────
print("Fetching Knox County road network from OSM...")
place = "Knox County, Tennessee, USA"
G_osm = ox.graph_from_place(place, network_type="drive")
osm_nodes, osm_edges = ox.graph_to_gdfs(G_osm)

print(f"OSM Nodes: {len(osm_nodes):,}  |  OSM Edges: {len(osm_edges):,}")
print("OSM Edge columns:", osm_edges.columns.tolist())


In [ ]:
# ── Fetch building footprints from OSM ───────────────────────────────────────
print("Fetching building footprints from OSM (this may take a moment)...")
buildings_gdf = ox.features_from_place(place, tags={"building": True})
buildings_gdf = buildings_gdf[buildings_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
buildings_gdf = buildings_gdf.to_crs(TARGET_CRS)
print(f"Buildings loaded: {len(buildings_gdf):,}")
print(buildings_gdf.head(2))


In [ ]:
# ── Zone-level morphology features ───────────────────────────────────────────
osm_edges_proj = osm_edges.to_crs(TARGET_CRS).copy()
osm_edges_proj["seg_length_m"] = osm_edges_proj.geometry.length

# Spatial join: edges → zones
edges_in_zones = gpd.sjoin(
    osm_edges_proj[["geometry", "seg_length_m"]].reset_index(drop=True),
    zones_gdf[["zone_id", "geometry"]],
    how="left", predicate="intersects"
)

morph_roads = edges_in_zones.groupby("zone_id").agg(
    n_segments=("seg_length_m", "count"),
    total_length_m=("seg_length_m", "sum"),
    avg_seg_length_m=("seg_length_m", "mean")
).reset_index()

# Spatial join: buildings → zones
buildings_in_zones = gpd.sjoin(
    buildings_gdf[["geometry"]].reset_index(drop=True),
    zones_gdf[["zone_id", "geometry"]],
    how="left", predicate="intersects"
)
morph_buildings = buildings_in_zones.groupby("zone_id").size().reset_index(name="n_buildings")

# Merge morphology features with zones
zones_gdf["area_km2"] = zones_gdf.geometry.area / 1e6
morph = zones_gdf[["zone_id", "area_km2"]].merge(morph_roads, on="zone_id", how="left")
morph = morph.merge(morph_buildings, on="zone_id", how="left").fillna(0)
morph["street_density_km_km2"] = morph["total_length_m"] / 1000 / morph["area_km2"]
morph["building_density_km2"]  = morph["n_buildings"] / morph["area_km2"]

print("Zone-level morphology features (top 10 by street density):")
print(morph.sort_values("street_density_km_km2", ascending=False).head(10).to_string(index=False))


---
## Section 9: Export Clean Datasets

In [ ]:
OUT_DIR = Path(r"C:\Users\ets\Desktop\GithubProjects\KnoxCity2Graph\outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Zones with morphology features
zones_out = zones_gdf.merge(morph.drop(columns=["area_km2"]), on="zone_id", how="left")
zones_out = zones_out.merge(od_zone_stats.reset_index(), on="zone_id", how="left")
zones_out.to_file(OUT_DIR / "zones_clean.gpkg", driver="GPKG")

# OD long table
od_long.to_csv(OUT_DIR / "od_long_2026.csv", index=False)

# Assignment with volumes
assign_merged.to_file(OUT_DIR / "assignment_with_volumes.gpkg", driver="GPKG")

# Morphology summary
morph.to_csv(OUT_DIR / "morphology_zone_features.csv", index=False)

print("Exported:")
print(f"  zones_clean.gpkg            — {len(zones_out):,} zones")
print(f"  od_long_2026.csv            — {len(od_long):,} OD pairs")
print(f"  assignment_with_volumes.gpkg — {len(assign_merged):,} links")
print(f"  morphology_zone_features.csv — {len(morph):,} zones")


---
## ✅ Step 1 & 2 Complete — What's Next?

**Step 3:** Regression analysis — does morphology explain productions, attractions, or assigned volumes?
- Join `zones_clean.gpkg` variables
- Run OLS / Poisson regressions: `production ~ street_density + building_density + ...`
- Visualise spatial residuals

**Step 4:** Build heterogeneous graph (zones ↔ OD edges ↔ morphology node features)

**Step 5:** GNN training (future work / paper extension)